In [ ]:
# ============================================================
# Cell 1: 环境检查 + 魔法方法 + 类别定义
# ============================================================
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

%pip install ultralytics -q

import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---- 自动检测设备 ----
if torch.cuda.is_available():
    DEVICE = 0
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    DEVICE = "cpu"
    print("无 GPU, 使用 CPU (会比较慢)")

# ---- 缺陷类别 (目前只有裂缝, 加数据后在这增删) ----
CLASS_NAMES = [
    "crack",                    # 0 裂缝
]

In [ ]:
# ============================================================
# Cell 2: 生成数据配置
# 跑之前确保 datasets/images/train/ 和 labels/train/ 下已放好数据
# ============================================================

from dataset import generate_data_yaml

generate_data_yaml(
    output_path="datasets/tunnel_defects.yaml",
    class_names=CLASS_NAMES,
)

In [ ]:
# ============================================================
# Cell 3: 训练 (所有可调参数都有中文注释)
# ============================================================

from ultralytics import YOLO

# ---- 模型选择 ----
# n = 3.2M 最快(先跑通) | s = 11.2M 推荐 | m = 25.9M 更准
model = YOLO("yolov8n.pt")

results = model.train(
    # ============ 数据 ============
    data="datasets/tunnel_defects.yaml",  # 数据配置文件路径

    # ============ 训练轮数 ============
    epochs=100,            # 总轮数，数据少可加到 200-300

    # ============ 批次大小 (显存不够就改小) ============
    batch=16,              # 8GB→8, 16GB→16, 24GB→32

    # ============ 输入尺寸 ============
    imgsz=640,             # 扣件等小目标可改 1280, 但更慢吃更多显存

    # ============ 学习率 ============
    lr0=0.01,              # 初始学习率
    lrf=0.01,              # 最终学习率 = lr0*lrf
    momentum=0.937,        # SGD 动量
    weight_decay=0.0005,   # L2 正则化, 防过拟合
    warmup_epochs=3.0,     # 预热轮数

    # ============ 数据增强 ============
    mosaic=1.0,            # 马赛克 (4张拼1张), 最后10轮自动关
    mixup=0.1,             # 两张图混合
    hsv_h=0.015,           # 色调抖动
    hsv_s=0.7,             # 饱和度抖动
    hsv_v=0.4,             # 明度抖动 (隧道光照变化大, 设高一点)
    degrees=15.0,          # 旋转 ±15°
    translate=0.1,         # 平移 10%
    scale=0.5,             # 缩放 50%
    fliplr=0.5,            # 左右翻转概率
    flipud=0.0,            # 上下翻转 (隧道一般不翻)
    erasing=0.4,           # 随机擦除, 模拟遮挡

    # ============ 早停 ============
    patience=50,           # 多少轮不涨就停

    # ============ 保存 ============
    save_period=5,         # 每5轮存一次 checkpoint

    # ============ 系统 ============
    device=DEVICE,          # Cell 1 自动检测: GPU→0, CPU→"cpu"
    workers=4,             # 数据加载线程
    amp=True,              # 混合精度, 省显存 (CPU 时自动忽略)
    seed=0,                # 随机种子, 0=随机

    # ============ 输出 ============
    project="runs",        # 输出根目录，ultralytics 会自动加 detect/ 子目录
    name="tunnel-det",     # 实验名
    exist_ok=True,         # 覆盖同名目录
    plots=True,            # 生成损失图等
)

print(f"\n训练完成!")
print(f"最佳: runs/detect/tunnel-det/weights/best.pt")
print(f"最后: runs/detect/tunnel-det/weights/last.pt")

In [ ]:
# ============================================================
# Cell 4: 训练曲线 (损失 + mAP + Precision + Recall)
# 每 5 轮标记数据点
# ============================================================

# 本次训练的实际输出目录（ultralytics 会在 project 下自动加 detect/ 子目录）
SAVE_DIR = "runs/detect/runs/train/tunnel-det"
results_csv = f"{SAVE_DIR}/results.csv"

if not Path(results_csv).exists():
    print(f"未找到 {results_csv}")
    print("请确认训练已完成，或修改 SAVE_DIR 指向正确的输出目录")
else:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"共 {len(df)} 轮数据")

    # ---- 三合一损失图 ----
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    titles = ["边界框损失 Box Loss", "分类损失 Class Loss", "DFL Loss"]
    cols = [("train/box_loss", "val/box_loss"),
            ("train/cls_loss", "val/cls_loss"),
            ("train/dfl_loss", "val/dfl_loss")]

    for ax, (train_col, val_col), title in zip(axes, cols, titles):
        ax.plot(df.index, df[train_col], label="训练", color="#1f77b4", lw=1)
        ax.plot(df.index, df[val_col], label="验证", color="#ff7f0e", lw=1)
        step = max(1, len(df) // 20)
        for i in range(0, len(df), step):
            ax.scatter(i, df[train_col].iloc[i], color="#1f77b4", s=18, zorder=5)
            ax.scatter(i, df[val_col].iloc[i], color="#ff7f0e", s=18, zorder=5)
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/loss_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ---- 指标图 ----
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    step = max(1, len(df) // 20)

    ax = axes[0]
    ax.plot(df.index, df["metrics/mAP50(B)"], color="#2ca02c", lw=1.5)
    [ax.scatter(i, df["metrics/mAP50(B)"].iloc[i], color="#2ca02c", s=18, zorder=5)
     for i in range(0, len(df), step)]
    ax.set_title("mAP@0.5 (越高越好)")
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(df.index, df["metrics/precision(B)"], color="#9467bd", lw=1.5, label="Precision")
    ax.plot(df.index, df["metrics/recall(B)"], color="#8c564b", lw=1.5, label="Recall")
    [ax.scatter(i, df["metrics/precision(B)"].iloc[i], color="#9467bd", s=15, zorder=5)
     for i in range(0, len(df), step)]
    [ax.scatter(i, df["metrics/recall(B)"].iloc[i], color="#8c564b", s=15, zorder=5)
     for i in range(0, len(df), step)]
    ax.set_title("Precision & Recall (越高越好)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(df.index, df["metrics/mAP50-95(B)"], color="#d62728", lw=1.5)
    [ax.scatter(i, df["metrics/mAP50-95(B)"].iloc[i], color="#d62728", s=18, zorder=5)
     for i in range(0, len(df), step)]
    ax.set_title("mAP@0.5:0.95 (越严格，越高越好)")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/metric_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# Cell 5: 验证 — 各类别指标
# ============================================================

model = YOLO(f"{SAVE_DIR}/weights/best.pt")
metrics = model.val()

print(f"mAP@0.5:       {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:  {metrics.box.map:.4f}")
print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")

print("\n各类别 AP@0.5:")
for i, ap in enumerate(metrics.box.ap50_per_class):
    name = CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"class_{i}"
    ap_val = ap.item() if hasattr(ap, "item") else ap
    bar = "█" * max(0, int(ap_val * 50))
    print(f"  {name:30s} {ap_val:.4f}  {bar}")

In [ ]:
# ============================================================
# Cell 6: 测试一张图
# ============================================================

from predict import predict_image
from IPython.display import Image as IPImage, display

# ---- 可调 ----
test_img = "test.jpg"   # 图片路径
conf_thr = 0.3           # 置信度: 漏检多→0.1, 误检多→0.5
iou_thr  = 0.5           # NMS: 框太多→0.3
# ------------

detections = predict_image(model, test_img, conf=conf_thr, iou=iou_thr, imgsz=640)

print(f"检测到 {len(detections)} 个缺陷:")
for d in detections:
    print(f"  [{d['confidence']:.2f}] {d['class']:20s} @ {d['bbox']}")

result_imgs = list(Path("runs/predict").glob("*.jpg"))
if result_imgs:
    display(IPImage(filename=str(result_imgs[0])))

In [ ]:
# ============================================================
# Cell 7: 导出 ONNX
# ============================================================

model.export(
    format="onnx",           # onnx / engine(TensorRT) / tflite
    imgsz=640,               # 输入尺寸
    simplify=True,            # 简化计算图
    half=False,               # FP16 (TensorRT 时推荐 True)
    opset=17,                 # ONNX opset
)
print(f"ONNX: {SAVE_DIR}/weights/best.onnx")

In [ ]:
# ============================================================
# 产出文件:
#   best.pt / last.pt     — 训练好的模型
#   best.onnx             — ONNX 部署文件
#   loss_curves.png       — 损失曲线
#   metric_curves.png     — mAP/Precision/Recall 曲线
# ============================================================
print("所有步骤完成!")